# `early_access_brazil` — Brazil interconnection sizing (2050)

**Scope:** a single, targeted fix -- `ELECTRICITY.avail_exterior` in
`Data/2050/early_access_brazil/{C3,C5}/Resources.csv`. `early_access_brazil` is otherwise
byte-identical to `early_access_2050` (verified: `Technologies.csv`, `Demands.csv`,
`Layers_in_out.csv`, `Misc_indep.json`, `Resources_indep.csv`, and `Misc.json`'s calibrated
`share_dispersion` all match exactly, cluster by cluster). The scenario must differ from
`early_access_2050` by this one parameter only.

**What was there before:** `C3` and `C5` already carried an ad-hoc `avail_exterior`
(100.0 and 55.0 GWh/y respectively, summing to 155.0 GWh/y) with a comment tracing it to
"cohérence d'ordre de grandeur avec les 133 ktCO2/an du MoU juillet 2024, pas mesuré" -- an
earlier, unmeasured placeholder from a July 2024 MoU-stage estimate. This notebook replaces it
with a sourced figure from the actual IDB program.

**New total: 113.8 GWh/y**, source: IDB (BID) **BO-L1238**, first-year program indicator --
**not a contractual commitment**. Split between `C3` and `C5` (the two clusters bordering Brazil
directly, per the existing scenario design -- `C1`, `C2`, `C4` are untouched, `C4` has no Brazil
border), proportional to each cluster's **diesel demand this import displaces**: `GENSET_DIESEL`
production in `no_transition_2050` (the scenario where diesel generation is actually built, since
`GENSET_DIESEL` is locked at 0 in `early_access` itself -- `no_transition`'s own build is the best
available proxy for "how much local diesel generation these two clusters would need absent solar
or import").

**Guardrail, per instruction:** `c_op_exterior`/`gwp_op_exterior` for `ELECTRICITY`
(0.0593 Meuro/GWh, 0.206486 ktCO2eq/GWh) live in the **shared** `00_INDEP/Resources_indep.csv` --
confirmed byte-identical across all 4 2050 scenarios, and **not touched here**. Touching it would
change the price/emissions factor for every scenario at once and break the cost/GWP gap
attribution between `early_access_brazil` and `early_access` (the whole point of this run is to
isolate the effect of the import route alone).


In [1]:
import pandas as pd

ROOT = "../../../EnergyScope_BO_nord_amazonia"

# Diesel-demand-displaced proxy: no_transition_2050's own GENSET_DIESEL production (the scenario
# where local diesel generation actually gets built, since early_access itself never builds any).
NT_2050_ASSETS_PATH = f"{ROOT}/case_studies/C1_C2_C3_C4_C5/norte_amazonia_no_transition_2050/outputs/regional_results/Assets.csv"

solve_info = pd.read_csv(
    f"{ROOT}/case_studies/C1_C2_C3_C4_C5/norte_amazonia_no_transition_2050/outputs/Solve_info.csv",
    sep=r"\t;\t", header=None, index_col=0, engine="python")
solve_result_num = int(float(solve_info.loc["solve_result_num", 1]))
assert solve_result_num == 0, (
    f"no_transition_2050: solve_result_num={solve_result_num} != 0 -- "
    f"refusing to size the Brazil import split from a non-optimal solve")

assets = pd.read_csv(NT_2050_ASSETS_PATH, sep=";")
assets["F_year"] = pd.to_numeric(assets["F_year"], errors="coerce")

BRAZIL_CLUSTERS = [3, 5]  # C3, C5 -- the two clusters bordering Brazil, per the existing scenario design
TOTAL_IMPORT_GWH = 113.8  # source: IDB BO-L1238, first-year program indicator, NOT a contractual commitment

genset_production = {}
for k in BRAZIL_CLUSTERS:
    row = assets[(assets["Regions"] == f"C{k}") & (assets["Technologies"] == "GENSET_DIESEL")]
    genset_production[k] = float(row["F_year"].values[0])

total_genset = sum(genset_production.values())
IMPORT_SPLIT_GWH = {k: TOTAL_IMPORT_GWH * genset_production[k] / total_genset for k in BRAZIL_CLUSTERS}

print(f"no_transition_2050 solve_result_num: {solve_result_num}")
print()
print("Diesel-demand-displaced proxy (no_transition_2050 GENSET_DIESEL production):")
for k in BRAZIL_CLUSTERS:
    print(f"  C{k}: {genset_production[k]:.4f} GWh/y ({genset_production[k]/total_genset*100:.2f}% of the C3+C5 total)")
print()
print(f"113.8 GWh/y split:")
for k in BRAZIL_CLUSTERS:
    print(f"  C{k}: {IMPORT_SPLIT_GWH[k]:.4f} GWh/y")
print(f"  sum = {sum(IMPORT_SPLIT_GWH.values()):.4f} GWh/y (should be 113.8)")
assert abs(sum(IMPORT_SPLIT_GWH.values()) - TOTAL_IMPORT_GWH) < 1e-6


no_transition_2050 solve_result_num: 0

Diesel-demand-displaced proxy (no_transition_2050 GENSET_DIESEL production):
  C3: 166.0176 GWh/y (66.79% of the C3+C5 total)
  C5: 82.5591 GWh/y (33.21% of the C3+C5 total)

113.8 GWh/y split:
  C3: 76.0039 GWh/y
  C5: 37.7961 GWh/y
  sum = 113.8000 GWh/y (should be 113.8)


## Deploy: overwrite `ELECTRICITY.avail_exterior` in `early_access_brazil`'s C3/C5 `Resources.csv`

Only the `ELECTRICITY` row's `avail_exterior` cell is touched in `C3` and `C5`; the stale
MoU-based `Comment` is replaced with the new sourcing note. `C1`, `C2`, `C4` are not touched.


In [2]:
COMMENT_TMPL = (
    "ELECTRICITY.avail_exterior = {value:.4f} GWh/y -- Brazil interconnection (early_access_brazil "
    "only). Total 113.8 GWh/y, source: IDB (BID) BO-L1238, first-year program indicator, NOT a "
    "contractual commitment. Split C3/C5 proportional to no_transition_2050's own GENSET_DIESEL "
    "production ({genset:.2f} GWh/y here, {total_genset:.2f} GWh/y C3+C5 total) -- the diesel "
    "generation this import displaces, since early_access itself never builds GENSET_DIESEL. "
    "c_op_exterior/gwp_op_exterior for ELECTRICITY are unchanged, defined once in the shared "
    "00_INDEP/Resources_indep.csv."
)

deployed_import = {}
for k in BRAZIL_CLUSTERS:
    path = f"{ROOT}/Data/2050/early_access_brazil/C{k}/Resources.csv"
    df = pd.read_csv(path, sep=";", index_col=0)
    df.index = df.index.str.strip()

    before = float(df.loc["ELECTRICITY", "avail_exterior"])
    df.loc["ELECTRICITY", "avail_exterior"] = IMPORT_SPLIT_GWH[k]
    df.loc["ELECTRICITY", "Comment"] = COMMENT_TMPL.format(
        value=IMPORT_SPLIT_GWH[k], genset=genset_production[k], total_genset=total_genset)
    df.to_csv(path, sep=";")

    check = pd.read_csv(path, sep=";", index_col=0)
    check.index = check.index.str.strip()
    after = float(check.loc["ELECTRICITY", "avail_exterior"])
    assert abs(after - IMPORT_SPLIT_GWH[k]) < 1e-9, (
        f"{path}: ELECTRICITY avail_exterior={after}, expected {IMPORT_SPLIT_GWH[k]}")
    deployed_import[k] = after
    print(f"C{k}: ELECTRICITY.avail_exterior {before:.4f} -> {after:.4f} GWh/y")

print()
print("ASSERT OK -- ELECTRICITY.avail_exterior redeployed and verified for C3/C5")


C3: ELECTRICITY.avail_exterior 100.0000 -> 76.0039 GWh/y
C5: ELECTRICITY.avail_exterior 55.0000 -> 37.7961 GWh/y

ASSERT OK -- ELECTRICITY.avail_exterior redeployed and verified for C3/C5


## Confirm `early_access_brazil` now differs from `early_access` by this one parameter only

In [3]:
EA_ROOT = f"{ROOT}/Data/2050/early_access"
EB_ROOT = f"{ROOT}/Data/2050/early_access_brazil"

for k in range(1, 6):
    ea = pd.read_csv(f"{EA_ROOT}/C{k}/Resources.csv", sep=";", index_col=0)
    eb = pd.read_csv(f"{EB_ROOT}/C{k}/Resources.csv", sep=";", index_col=0)
    ea.index = ea.index.str.strip()
    eb.index = eb.index.str.strip()

    diffs = []
    for resource in ea.index:
        for col in ["avail_local", "avail_exterior", "gwp_op_local", "c_op_local"]:
            va, vb = float(ea.loc[resource, col]), float(eb.loc[resource, col])
            if abs(va - vb) > 1e-9:
                diffs.append((resource, col, va, vb))

    if k == 1:
        # C1 has its own SIN-logic comment/value in BOTH scenarios already (not Brazil-specific,
        # not touched here) -- expect zero numeric diffs.
        assert not diffs, f"C1: unexpected numeric diffs {diffs}"
        print(f"C1: no numeric diffs (expected -- C1's note is scenario-independent, untouched)")
    elif k in (3, 5):
        assert len(diffs) == 1 and diffs[0][0] == "ELECTRICITY" and diffs[0][1] == "avail_exterior", (
            f"C{k}: expected exactly one diff (ELECTRICITY.avail_exterior), got {diffs}")
        print(f"C{k}: exactly one diff -- ELECTRICITY.avail_exterior {diffs[0][2]:.4f} -> {diffs[0][3]:.4f} GWh/y")
    else:
        assert not diffs, f"C{k}: unexpected numeric diffs {diffs}"
        print(f"C{k}: no numeric diffs (expected -- not part of the Brazil route)")

print()
print("ASSERT OK -- early_access_brazil differs from early_access by ELECTRICITY.avail_exterior "
      "in C3/C5 only, nothing else")


C1: no numeric diffs (expected -- C1's note is scenario-independent, untouched)
C2: no numeric diffs (expected -- not part of the Brazil route)
C3: exactly one diff -- ELECTRICITY.avail_exterior 0.0000 -> 76.0039 GWh/y
C4: no numeric diffs (expected -- not part of the Brazil route)
C5: exactly one diff -- ELECTRICITY.avail_exterior 0.0000 -> 37.7961 GWh/y

ASSERT OK -- early_access_brazil differs from early_access by ELECTRICITY.avail_exterior in C3/C5 only, nothing else


## Reprint `reg_resources.dat` from the deployed catalog (no solve)

Same mechanism used for every fix this session: instantiate `Esmc`, read the just-deployed
`Data/2050/early_access_brazil/` catalog, apply the same pre-solve preprocessing
`scripts/run.py` applies (`ft_to_drop`, `PV_UTILITY`/`BATT_LI` unlock -- `early_access_brazil`
matches the `norte_amazonia_early_access_` prefix check), reuse the shared typical-day cache
(`algo='read'`, no AMPL call), then `print_data(indep=True)`. **No `set_esom()`/`solve_esom()`
call -- no AMPL solve is launched.**


In [4]:
import sys
sys.path.insert(0, r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
from pathlib import Path
from esmc import Esmc

case_study = "norte_amazonia_early_access_brazil_2050"
FT_TO_DROP = ['BIOMASS_TO_GASOLINE', 'BIOMASS_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_DIESEL',
              'POWER_TO_GASOLINE', 'POWER_TO_DIESEL', 'H2_TO_GASOLINE', 'H2_TO_DIESEL']
AMPL_PATH = r'C:\Users\valen\AMPL'  # unused by algo='read'

config = {'case_study': case_study, 'comment': 'Brazil interconnection reprint (no solve)',
          'regions_names': ['C1', 'C2', 'C3', 'C4', 'C5'],
          'gwp_limit_overall': None, 're_share_primary': None, 'f_perc': True,
          'year': 2050, 'scenario': 'early_access_brazil'}

my_model = Esmc(config, nbr_td=16)
current_project = Path(r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
my_model.project_dir = current_project
my_model.dat_dir = current_project / 'case_studies' / my_model.space_id / '00_td_dat'
my_model.cs_dir = current_project / 'case_studies' / my_model.space_id / case_study
my_model.dat_dir.mkdir(parents=True, exist_ok=True)
my_model.cs_dir.mkdir(parents=True, exist_ok=True)

my_model.read_data_indep()
my_model.init_regions()

my_model.ref_region.data['Technologies'] = my_model.ref_region.data['Technologies'].drop(index=FT_TO_DROP)
my_model.data_indep['Layers_in_out'] = my_model.data_indep['Layers_in_out'].drop(index=FT_TO_DROP)
for r_code, region in my_model.regions.items():
    region.data['Technologies'] = region.data['Technologies'].drop(index=FT_TO_DROP)

assert case_study.startswith('norte_amazonia_early_access_')
for r_code, region in my_model.regions.items():
    region.data['Technologies'].loc['PV_UTILITY', 'f_max'] = 1e15
    region.data['Technologies'].loc['BATT_LI', 'f_max'] = 1e15

# Pre-print sanity check
for k in BRAZIL_CLUSTERS:
    region_code = f"C{k}"
    in_memory = float(my_model.regions[region_code].data['Resources'].loc['ELECTRICITY', 'avail_exterior'])
    expected = deployed_import[k]
    assert abs(in_memory - expected) < 1e-9, (
        f"{region_code}: in-memory ELECTRICITY avail_exterior={in_memory} after init_regions(), "
        f"expected {expected} (deployed CSV)")

my_model.init_ta(algo='read', ampl_path=AMPL_PATH)
my_model.print_td_data()
my_model.print_data(indep=True)

print(f"OK -- reg_resources.dat reprinted for {case_study} at {my_model.cs_dir} "
      f"(pre-print in-memory check passed, no solve)")

# Text spot-check of the reprinted .dat itself
dat_path = my_model.cs_dir / "reg_resources.dat"
lines = dat_path.read_text(encoding="utf-8").splitlines()
AVAIL_EXT_FIELD_INDEX = 3  # REGION RESOURCE avail_local avail_exterior gwp_op_local c_op_local
for k in BRAZIL_CLUSTERS:
    row = next((l for l in lines if l.split() and l.split()[0] == f"C{k}" and l.split()[1] == "ELECTRICITY"), None)
    assert row is not None, f"reg_resources.dat: no ELECTRICITY row found for C{k}"
    printed = float(row.split()[AVAIL_EXT_FIELD_INDEX])
    assert abs(printed - deployed_import[k]) < 1e-6, (
        f"C{k}: printed avail_exterior={printed}, expected {deployed_import[k]}")
print(f"OK -- reg_resources.dat text spot-check passed for C3/C5")


[INFO    ] (read_data_indep): Read indep data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\early_access_brazil\00_INDEP


[INFO    ] (init_regions): Initialising regions: C1, C2, C3, C4, C5


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\early_access_brazil\02_REF_REGION


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\early_access_brazil\C1


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\early_access_brazil\C2


...

[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_td_data): Printing TD data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050\reg_16TD.dat


[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_data): Printing regional data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050


OK -- reg_resources.dat reprinted for norte_amazonia_early_access_brazil_2050 at C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050 (pre-print in-memory check passed, no solve)
OK -- reg_resources.dat text spot-check passed for C3/C5
